<a href="https://colab.research.google.com/github/Shreyas0744/Driver-Drowsiness-Detection-System/blob/main/Driver_Drowsiness_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install dependencies
!pip install imutils
!pip install dlib

# 2. Download the pre-trained facial landmark predictor
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bunzip2 shape_predictor_68_face_landmarks.dat.bz2

import cv2
import dlib
import numpy as np
from scipy.spatial import distance as dist
from imutils import face_utils
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
import base64

--2026-02-24 09:39:27--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-02-24 09:39:27--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  37.6MB/s    in 1.6s    

2026-02-24 09:39:29 (37.6 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]



In [2]:
# Define the Eye Aspect Ratio (EAR) function
def eye_aspect_ratio(eye):
    # Compute the euclidean distances between the two sets of vertical eye landmarks
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    # Compute the euclidean distance between the horizontal eye landmark
    C = dist.euclidean(eye[0], eye[3])
    # Compute the EAR
    ear = (A + B) / (2.0 * C)
    return ear

# JavaScript to handle the Webcam in Colab
def video_stream():
  js = Javascript('''
    var video;
    var canvas;
    var div;
    var capture = null;

    async function startVideo() {
      video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(video);
      video.srcObject = stream;
      await video.play();

      canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
    }

    async function takePhoto() {
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
  display(js)

# Initialize dlib's face detector and shape predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Define thresholds
EYE_AR_THRESH = 0.25 # If EAR is less than this, eyes are likely closed
EYE_AR_CONSEC_FRAMES = 15 # Eyes must be closed for 15 frames to trigger alert

In [4]:
# Main Logic loop
def run_drowsiness_detection():
    video_stream()
    eval_js('startVideo()')

    counter = 0
    while True:
        # Capture frame-by-frame from JS webcam
        img_str = eval_js('takePhoto()')
        header, data = img_str.split(',')
        binary = base64.b64decode(data)
        image = cv2.imdecode(np.frombuffer(binary, np.uint8), -1)

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        rects = detector(gray, 0)

        for rect in rects:
            shape = predictor(gray, rect)
            shape = face_utils.shape_to_np(shape)

            # Extract left and right eye coordinates
            leftEye = shape[36:42]
            rightEye = shape[42:48]

            leftEAR = eye_aspect_ratio(leftEye)
            rightEAR = eye_aspect_ratio(rightEye)
            ear = (leftEAR + rightEAR) / 2.0

            # Draw eye contours for visualization
            leftEyeHull = cv2.convexHull(leftEye)
            rightEyeHull = cv2.convexHull(rightEye)
            cv2.drawContours(image, [leftEyeHull], -1, (0, 255, 0), 1)
            cv2.drawContours(image, [rightEyeHull], -1, (0, 255, 0), 1)

            if ear < EYE_AR_THRESH:
                counter += 1
                if counter >= EYE_AR_CONSEC_FRAMES:
                    cv2.putText(image, "DROWSINESS ALERT!", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            else:
                counter = 0

        # Note: In Colab, we can't use live video windows, so we clear and show frames
        from IPython.display import clear_output
        clear_output(wait=True)
        cv2_imshow(image)

# Start the detection
# run_drowsiness_detection()